# The Ising Spin Model VQE

**Lecture Exercises — Quantum Computing**

The **Ising model** is one of the most important models in statistical mechanics and has deep connections to quantum computing. Its optimisation landscape maps directly onto **Quadratic Unconstrained Binary Optimisation (QUBO)** problems, which are the native problem class for quantum annealers (e.g., D-Wave) and are widely targeted by variational quantum algorithms.

In this notebook we will:

1. **Define the Ising spin model** and its energy function
2. **Enumerate all spin configurations** and find the ground state by brute force
3. **Build a variational quantum circuit** to find the ground state
4. **Optimise the circuit parameters** using a classical optimiser
5. **Explore the optimisation landscape** and the problem of local minima
6. **Generalise to larger systems** and connect to QUBO

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from itertools import product as cartesian_product

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

## 2. The Ising Spin Model

### The Hamiltonian

The Ising model describes a system of interacting spins $s_i \in \{+1, -1\}$ sitting on the vertices of a graph. In the absence of external fields, the energy (Hamiltonian) is:

$$H = -\sum_{\langle i,j \rangle} J_{ij}\, s_i\, s_j$$

where:
- $s_i = +1$ (“spin up”) or $s_i = -1$ (“spin down”)
- $J_{ij}$ is the **coupling strength** between spins $i$ and $j$
- The sum runs over **nearest-neighbour pairs** $\langle i,j \rangle$

The sign of $J_{ij}$ determines the nature of the interaction:
- $J_{ij} > 0$: **ferromagnetic** — neighbouring spins prefer to align
- $J_{ij} < 0$: **antiferromagnetic** — neighbouring spins prefer to anti-align

### Connection to qubits

On a quantum computer, each spin maps naturally to a qubit:
- Spin up ($s_i = +1$) $\leftrightarrow$ $|0\rangle$ (eigenvalue $+1$ of the Pauli-$Z$ operator)
- Spin down ($s_i = -1$) $\leftrightarrow$ $|1\rangle$ (eigenvalue $-1$ of Pauli-$Z$)

The classical Ising energy becomes a **quantum Hamiltonian** by replacing $s_i \to Z_i$:

$$H = -\sum_{\langle i,j \rangle} J_{ij}\, Z_i Z_j$$

Finding the **ground state** (minimum energy configuration) of this Hamiltonian is the central optimisation problem.

### Connection to QUBO

Any QUBO problem of the form $\min_x \sum_{ij} Q_{ij} x_i x_j$ with $x_i \in \{0, 1\}$ can be mapped to an Ising model via the substitution $x_i = (1 - s_i)/2$. This means solving the Ising ground-state problem is equivalent to solving a QUBO — and both are NP-hard in general.

## 3. A Concrete Example: The 3-Spin Ising Chain

Let’s study a simple 3-spin chain where:
- Spin $s_1$ is **fixed** at $+1$ (spin up)
- The couplings are $J_{12} = -1$ (antiferromagnetic) and $J_{23} = +1$ (ferromagnetic)

The energy is:

$$E(s_1, s_2, s_3) = -J_{12}\, s_1 s_2 - J_{23}\, s_2 s_3 = s_1 s_2 - s_2 s_3$$

Since $s_1 = +1$ is fixed, we only need to optimise over $s_2$ and $s_3$.

In [ ]:
# Define the coupling constants
J = [-1, +1]  # J12 = -1 (antiferromagnetic), J23 = +1 (ferromagnetic)

# Enumerate all spin configurations (s1 fixed at +1)
print("All spin configurations (s1 = +1 fixed):\n")
print(f"{'[s1, s2, s3]':>15s}  {'Energy':>8s}")
print("-" * 30)

all_energies = {}
for s2, s3 in cartesian_product([+1, -1], repeat=2):
    s1 = +1  # fixed
    E = -J[0] * s1 * s2 - J[1] * s2 * s3
    config = (s1, s2, s3)
    all_energies[config] = E

E_min = min(all_energies.values())
for config, E in sorted(all_energies.items(), key=lambda x: x[1]):
    marker = '  <-- GROUND STATE' if E == E_min else ''
    print(f"  [{config[0]:+d}, {config[1]:+d}, {config[2]:+d}]  {E:+8.1f}{marker}")

print(f"\nGround state energy: {E_min}")
print(f"Ground state: {[c for c, e in all_energies.items() if e == E_min]}")

The ground state is $[s_1, s_2, s_3] = [+1, -1, -1]$ with energy $E = -2$.

**Physical interpretation:**
- $J_{12} = -1$ (antiferromagnetic): $s_1$ and $s_2$ prefer to be anti-aligned $\to$ $s_2 = -1$ ✔
- $J_{23} = +1$ (ferromagnetic): $s_2$ and $s_3$ prefer to be aligned $\to$ $s_3 = -1$ ✔

Both couplings are satisfied simultaneously, giving the lowest possible energy.

## 4. The Quantum Hamiltonian

We now express the Ising energy as a quantum operator using Pauli-$Z$ matrices. With couplings $J_{12} = -1$ and $J_{23} = +1$:

$$H = -J_{12}\, Z_0 Z_1 - J_{23}\, Z_1 Z_2 = Z_0 Z_1 - Z_1 Z_2$$

In Qiskit, we represent this using `SparsePauliOp`.

In [ ]:
def ising_hamiltonian(n, couplings):
    """Build the Ising chain Hamiltonian as a SparsePauliOp.
    
    H = -sum_i J_i * Z_i * Z_{i+1}
    
    Parameters:
        n: number of spins
        couplings: list of coupling constants [J_01, J_12, ..., J_{n-2,n-1}]
    
    Returns:
        SparsePauliOp
    """
    terms = []
    for i, J_val in enumerate(couplings):
        pauli = ['I'] * n
        pauli[i] = 'Z'
        pauli[i + 1] = 'Z'
        # SparsePauliOp convention: qubit 0 is rightmost
        terms.append((''.join(reversed(pauli)), -J_val))
    return SparsePauliOp.from_list(terms)


# Build the 3-spin Hamiltonian
H = ising_hamiltonian(3, J)
print("Ising Hamiltonian:")
print(H)

# Verify: the diagonal gives the energy for each computational basis state
print("\nEnergy for each basis state:")
diag = np.diag(H.to_matrix().real)
for idx, E_val in enumerate(diag):
    bitstring = format(idx, '03b')
    spins = [1 - 2*int(b) for b in bitstring[::-1]]  # [s1, s2, s3]
    print(f"  |{bitstring}> -> spins [{spins[0]:+d}, {spins[1]:+d}, {spins[2]:+d}]: E = {E_val:+.0f}")

# Exact ground state energy
eigenvalues = np.sort(np.linalg.eigvalsh(H.to_matrix()).real)
print(f"\nEnergy spectrum: {eigenvalues}")
print(f"Ground state energy: {eigenvalues[0]:.1f}")

## 5. Variational Quantum Circuit for Ground-State Search

Rather than enumerating all $2^n$ configurations (which is exponentially expensive), we use a **variational quantum circuit** (ansatz) to search for the ground state.

### Strategy

1. **Fix qubit 0** in $|0\rangle$ (spin up, $s_1 = +1$).
2. **Parameterise qubits 1 and 2** with rotation gates.
3. **Measure** $\langle H \rangle = -J_{12}\langle Z_0 Z_1\rangle - J_{23}\langle Z_1 Z_2\rangle$.
4. **Optimise** the rotation angles to minimise $\langle H \rangle$.

Since the Ising Hamiltonian is diagonal in the $Z$ basis, a single $R_y(\theta)$ rotation per free qubit suffices to explore all product states in the $Z$ eigenbasis:
- $R_y(0)|0\rangle = |0\rangle$ (spin up, $s = +1$)
- $R_y(\pi)|0\rangle = |1\rangle$ (spin down, $s = -1$)

For a fully general single-qubit state, we can use the general rotation $\text{Rot}(\phi, \theta, \omega) = R_z(\omega)\, R_y(\theta)\, R_z(\phi)$, which matches the original PennyLane parameterisation.

In [ ]:
def ising_ansatz(n, params, fixed_qubits=None):
    """Build a variational ansatz for the Ising model.
    
    Each free qubit gets a general Rot(phi, theta, omega) rotation.
    Fixed qubits remain in |0> (spin up).
    
    Parameters:
        n: number of qubits
        params: dict mapping qubit index to (phi, theta, omega) angles
        fixed_qubits: list of qubit indices that are fixed at |0>
    
    Returns:
        QuantumCircuit
    """
    if fixed_qubits is None:
        fixed_qubits = []
    
    qc = QuantumCircuit(n)
    for qubit in range(n):
        if qubit not in fixed_qubits and qubit in params:
            phi, theta, omega = params[qubit]
            qc.rz(phi, qubit)
            qc.ry(theta, qubit)
            qc.rz(omega, qubit)
    
    return qc


def compute_energy(ansatz, hamiltonian, n):
    """Compute the expectation value <H> for a given ansatz."""
    sv = Statevector.from_label('0' * n).evolve(ansatz)
    return sv.expectation_value(hamiltonian).real


# Test: the |000> state (all spin up) should give E = 0
qc_test = ising_ansatz(3, {}, fixed_qubits=[0])
E_test = compute_energy(qc_test, H, 3)
print(f"Energy of |000> (all up): {E_test:.4f}  (expected: 0)")

# Test: |110> in Qiskit = qubit 1 and 2 in |1> -> spins [+1, -1, -1], should give E = -2
qc_test2 = ising_ansatz(3, {1: (0, np.pi, 0), 2: (0, np.pi, 0)}, fixed_qubits=[0])
E_test2 = compute_energy(qc_test2, H, 3)
print(f"Energy of spins [+1,-1,-1]: {E_test2:.4f}  (expected: -2)")

# Draw the ansatz
qc_draw = ising_ansatz(3, {1: (0.5, 1.0, 0.3), 2: (0.7, 0.8, 0.2)}, fixed_qubits=[0])
qc_draw.draw('mpl')

### Exercise 1: Test known spin configurations

Before optimising, verify that the circuit correctly reproduces the energy of known spin configurations.

**(a)** Use the ansatz to prepare the state $[s_1, s_2, s_3] = [+1, -1, -1]$ and confirm the energy is $-2$.

**(b)** Prepare $[s_1, s_2, s_3] = [+1, +1, -1]$ and confirm the energy is $+2$.

**(c)** What rotation angles correspond to spin up ($+1$) and spin down ($-1$)?

In [ ]:
# Exercise 1: Your code here


## 6. Optimising the Variational Circuit

We now use a classical optimiser to find the rotation angles that minimise $\langle H \rangle$. The variational loop is:

1. Choose initial angles (randomly).
2. Build the ansatz circuit with those angles.
3. Compute the energy $\langle H \rangle$.
4. Update the angles to reduce the energy.
5. Repeat until convergence.

In [ ]:
def run_vqe_ising(n, couplings, fixed_qubits=None, method='COBYLA',
                   maxiter=200, seed=None, use_general_rot=True):
    """Run VQE to find the Ising ground state.
    
    Parameters:
        n: number of spins
        couplings: list of coupling constants
        fixed_qubits: list of fixed qubit indices
        method: optimiser method
        maxiter: max iterations
        seed: random seed for initial parameters
        use_general_rot: if True, use Rot(phi,theta,omega); if False, use only Ry(theta)
    
    Returns:
        dict with results
    """
    if fixed_qubits is None:
        fixed_qubits = []
    
    H_local = ising_hamiltonian(n, couplings)
    free_qubits = [q for q in range(n) if q not in fixed_qubits]
    params_per_qubit = 3 if use_general_rot else 1
    n_params = len(free_qubits) * params_per_qubit
    
    rng = np.random.default_rng(seed)
    x0 = rng.uniform(0, np.pi, n_params)
    
    history = {'energies': [], 'params': []}
    
    def cost_function(param_values):
        params = {}
        for i, qubit in enumerate(free_qubits):
            if use_general_rot:
                params[qubit] = (param_values[3*i], param_values[3*i+1], param_values[3*i+2])
            else:
                params[qubit] = (0, param_values[i], 0)
        
        qc = ising_ansatz(n, params, fixed_qubits)
        energy = compute_energy(qc, H_local, n)
        history['energies'].append(energy)
        history['params'].append(param_values.copy())
        return energy
    
    result = minimize(cost_function, x0, method=method,
                      options={'maxiter': maxiter})
    
    # Extract final state
    final_params = {}
    for i, qubit in enumerate(free_qubits):
        if use_general_rot:
            final_params[qubit] = (result.x[3*i], result.x[3*i+1], result.x[3*i+2])
        else:
            final_params[qubit] = (0, result.x[i], 0)
    
    qc_final = ising_ansatz(n, final_params, fixed_qubits)
    sv_final = Statevector.from_label('0' * n).evolve(qc_final)
    probs = sv_final.probabilities_dict()
    most_likely = max(probs, key=probs.get)
    spins = [1 - 2*int(b) for b in most_likely[::-1]]
    
    return {
        'optimal_energy': result.fun,
        'optimal_params': result.x,
        'final_params': final_params,
        'history': history,
        'most_likely_state': most_likely,
        'spins': spins,
        'converged': result.success,
        'n_evaluations': len(history['energies']),
    }

In [ ]:
# Run the optimisation
np.random.seed(56)
result = run_vqe_ising(3, J, fixed_qubits=[0], method='COBYLA',
                        maxiter=200, seed=56, use_general_rot=True)

E_exact = min(np.linalg.eigvalsh(H.to_matrix()).real)

print(f"3-Spin Ising Chain VQE Results")
print(f"{'='*45}")
print(f"Couplings:        J12 = {J[0]:+d}, J23 = {J[1]:+d}")
print(f"Fixed spins:      s1 = +1 (qubit 0)\n")
print(f"Optimal energy:   {result['optimal_energy']:.6f}")
print(f"Exact energy:     {E_exact:.6f}")
print(f"Error:            {abs(result['optimal_energy'] - E_exact):.6f}\n")
print(f"Spin configuration: {result['spins']}")
print(f"Evaluations:      {result['n_evaluations']}")

print(f"\nOptimal rotation angles:")
for qubit, (phi, theta, omega) in result['final_params'].items():
    print(f"  Qubit {qubit}: (phi, theta, omega) = ({phi:.4f}, {theta:.4f}, {omega:.4f})")

In [ ]:
# Plot convergence
fig, ax = plt.subplots(figsize=(10, 5))
energies = result['history']['energies']
ax.plot(energies, 'b-', linewidth=1, alpha=0.7)
ax.axhline(y=E_exact, color='red', linestyle='--', linewidth=2,
           label=f'Exact ground state: {E_exact:.1f}')
ax.set_xlabel('Optimiser iteration', fontsize=12)
ax.set_ylabel('Energy $\\langle H \\rangle$', fontsize=12)
ax.set_title('VQE Convergence for 3-Spin Ising Chain', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final energy: {energies[-1]:.6f} (target: {E_exact:.1f})")

The optimiser converges to $E = -2$, corresponding to the spin configuration $[+1, -1, -1]$. The optimal angles are $(\phi, \theta, \omega) \approx (0, \pi, 0)$ for both free qubits, rotating them from $|0\rangle$ (spin up) to $|1\rangle$ (spin down).

## 7. The Local Minima Problem

The Ising model’s optimisation landscape is **non-convex**: there can be multiple local minima that trap gradient-based optimisers. Let’s explore this by running the optimisation from many random starting points.

In [ ]:
# Run VQE with many different random seeds
n_trials = 50
results_all = []

for seed in range(n_trials):
    res = run_vqe_ising(3, J, fixed_qubits=[0], method='COBYLA',
                         maxiter=300, seed=seed, use_general_rot=True)
    results_all.append(res)

final_energies = [r['optimal_energy'] for r in results_all]
found_ground = [abs(e - E_exact) < 0.01 for e in final_energies]

print(f"Local Minima Analysis ({n_trials} random starting points)\n")
print(f"Ground state energy: {E_exact:.1f}")
print(f"Found global minimum: {sum(found_ground)} / {n_trials} "
      f"({100*sum(found_ground)/n_trials:.0f}%)\n")

from collections import Counter
energy_counts = Counter([round(e, 1) for e in final_energies])
for energy, count in sorted(energy_counts.items()):
    bar = '#' * count
    label = ' <-- global' if abs(energy - E_exact) < 0.1 else ''
    print(f"  E = {energy:+5.1f}: {count:3d} runs  {bar}{label}")

In [ ]:
# Visualise convergence trajectories coloured by outcome
fig, ax = plt.subplots(figsize=(12, 6))

for res in results_all:
    en = res['history']['energies']
    final_E = res['optimal_energy']
    color = 'green' if abs(final_E - E_exact) < 0.01 else 'red'
    alpha = 0.5 if color == 'green' else 0.3
    ax.plot(en, color=color, linewidth=0.8, alpha=alpha)

ax.axhline(y=E_exact, color='black', linestyle='--', linewidth=2)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='green', linewidth=2, label='Found global minimum'),
    Line2D([0], [0], color='red', linewidth=2, label='Stuck in local minimum'),
    Line2D([0], [0], color='black', linestyle='--', linewidth=2, label=f'Ground state (E={E_exact:.0f})'),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel('Optimiser iteration', fontsize=12)
ax.set_ylabel('Energy', fontsize=12)
ax.set_title('VQE Convergence: Multiple Random Initialisations', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Visualising the Energy Landscape

Since qubit 0 is fixed, we have two free parameters (the $\theta$ angles for $R_y$ on qubits 1 and 2). This lets us plot the full energy landscape.

In [ ]:
n_points = 100
theta1_range = np.linspace(0, 2*np.pi, n_points)
theta2_range = np.linspace(0, 2*np.pi, n_points)

energy_landscape = np.zeros((n_points, n_points))
for i, t1 in enumerate(theta1_range):
    for j, t2 in enumerate(theta2_range):
        qc = ising_ansatz(3, {1: (0, t1, 0), 2: (0, t2, 0)}, fixed_qubits=[0])
        energy_landscape[j, i] = compute_energy(qc, H, 3)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.contourf(theta1_range / np.pi, theta2_range / np.pi, energy_landscape,
                  levels=30, cmap='RdYlBu_r')
plt.colorbar(im, ax=ax, label='Energy $\\langle H \\rangle$')

# Mark the classical spin configurations
configs = [
    ('$[+1,+1,+1]$\nE=0', 0, 0),
    ('$[+1,+1,-1]$\nE=+2', 0, 1),
    ('$[+1,-1,+1]$\nE=0', 1, 0),
    ('$[+1,-1,-1]$\nE=-2', 1, 1),
]
for label, t1_pi, t2_pi in configs:
    ax.plot(t1_pi, t2_pi, 'ko', markersize=10)
    ax.annotate(label, xy=(t1_pi, t2_pi), fontsize=8, xytext=(8, 8),
                textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_xlabel('$\\theta_1 / \\pi$ (qubit 1 rotation)', fontsize=12)
ax.set_ylabel('$\\theta_2 / \\pi$ (qubit 2 rotation)', fontsize=12)
ax.set_title('Energy Landscape of the 3-Spin Ising Chain', fontsize=13)
plt.tight_layout()
plt.show()

The global minimum (dark blue) is at (theta_1, theta_2) = (pi, pi), corresponding to spins [+1, -1, -1] with E = -2.


## 9. Generalisation: Larger Ising Chains

The approach generalises to longer chains. Let’s test on 5 and 6 spins with random couplings.

In [ ]:
np.random.seed(42)

for n_spins in [5, 6]:
    couplings = np.random.choice([-1, +1], n_spins - 1).tolist()
    H_n = ising_hamiltonian(n_spins, couplings)
    E_exact_n = min(np.linalg.eigvalsh(H_n.to_matrix()).real)
    
    # VQE with multiple restarts
    best_vqe = None
    for seed in range(20):
        res = run_vqe_ising(n_spins, couplings, fixed_qubits=[], method='COBYLA',
                             maxiter=500, seed=seed, use_general_rot=False)
        if best_vqe is None or res['optimal_energy'] < best_vqe['optimal_energy']:
            best_vqe = res
    
    print(f"{n_spins}-spin chain, J = {couplings}")
    print(f"  Exact:  E = {E_exact_n:.1f}")
    print(f"  VQE:    E = {best_vqe['optimal_energy']:.4f}, spins = {best_vqe['spins']}")
    print(f"  Error:  {abs(best_vqe['optimal_energy'] - E_exact_n):.4f}\n")

---
## 10. Exercises

### Exercise 2: Different initial conditions and optimisers

Due to the complexity of the Ising model’s optimisation space, it is easy to find local minima.

**(a)** Run the 3-spin VQE with 100 different random seeds using the general rotation and count how many find the global minimum.

**(b)** Repeat using only $R_y(\theta)$ rotations. Does the success rate change?

**(c)** Try different optimisers: `COBYLA`, `Nelder-Mead`, `Powell`. Which finds the global minimum most reliably?


### Exercise 3: Frustrated systems

**(a)** Consider an antiferromagnetic triangle: 3 spins with all couplings $J_{ij} = -1$ and periodic boundary conditions. Build the Hamiltonian (including a $Z_2 Z_0$ term) and find the ground state by brute force.

**(b)** Is the ground state **frustrated** (can all couplings be simultaneously satisfied)?

**(c)** What is the ground-state degeneracy?


### Exercise 4: External magnetic field

Add a transverse field to the Ising model:

$$H = -\sum_{\langle i,j \rangle} J_{ij} Z_i Z_j - h \sum_i X_i$$

**(a)** Modify `ising_hamiltonian` to include the field term.

**(b)** The ground state is no longer a computational basis state. Why?

**(c)** Run VQE for $J = [-1, 1]$ and $h = 0.5$. Is the $R_y$-only ansatz sufficient?


### Exercise 5: Scaling

**(a)** Run VQE for Ising chains of length $n = 3, 4, 5, 6, 7, 8$ with random couplings. Plot the success rate (fraction of restarts finding the global minimum) vs $n$.

**(b)** At what system size does the variational approach begin to struggle?

In [ ]:
# Exercise 2: Your code here


In [ ]:
# Exercise 3: Your code here


In [ ]:
# Exercise 4: Your code here


In [ ]:
# Exercise 5: Your code here
